# Time-budgeted LLM pretraining - Kaggle **T4 x2** (v3.1)
Same model, data and hyperparameters (global batch 65,536 tokens). v3.1 fixes the **CUDA OOM** seen at step 2 (per-GPU micro-batch 8 -> 4, accumulation 4 -> 8) and the notebook now auto-halves the micro-batch if it ever OOMs again.

Run cells in order. Settings: GPU T4 x2, Internet ON.

In [1]:
import subprocess, sys
print(sys.version)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
need = [m for m in ("transformers","datasets","tokenizers","huggingface_hub","psutil","safetensors")
        if __import__("importlib.util").util.find_spec(m) is None]
if need: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
GPU 0: Tesla T4 (UUID: GPU-664ac5bd-127b-2626-1320-88f7d491319f)
GPU 1: Tesla T4 (UUID: GPU-f8f7300c-d964-a412-c5c3-656bc4ab0d4d)



In [2]:
%%writefile /kaggle/working/train_pipeline.py
"""
Time-budgeted LLM pretraining for Kaggle T4 x2 (DDP, fp16 AMP). Llama-style ~100M params. (v3: hang-proof)

  python train_pipeline.py prep                                      # tokenizer + pre-tokenized data (CPU only)
  torchrun --standalone --nproc_per_node=2 train_pipeline.py train   # DDP training on both GPUs

Outputs (in /kaggle/working/final_model): model weights + tokenizer + training_summary.json
Reload anywhere:  AutoModelForCausalLM.from_pretrained(path); AutoTokenizer.from_pretrained(path)
"""
import os, sys, time, math, json, queue, threading, shutil, itertools, contextlib
from datetime import timedelta
from pathlib import Path
import numpy as np
import torch
import torch.distributed as dist

SMOKE = os.environ.get("SMOKE", "0") == "1"      # tiny CPU test of the whole pipeline
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")   # patient with slow HF responses
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

# Session clock: the notebook exports the kernel start time so torchrun children share one clock.
if os.environ.get("SESSION_T0"):
    T0 = float(os.environ["SESSION_T0"])
else:
    try:
        import psutil; T0 = psutil.Process().create_time()
    except Exception:
        T0 = time.time()

RANK = int(os.environ.get("RANK", 0))
IS_MAIN = RANK == 0
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("./work")
DATA = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else WORK / "data"   # not persisted -> small output
CKPT = WORK / "ckpt"
FINAL = WORK / "final_model"

# ---- time budget (Kaggle commit = 12h hard limit; ~30 min is spent persisting output) ----
WALL_LIMIT_H = 10.5          # whole session (incl. data prep) must be finished by this point
FINAL_RESERVE_MIN = 12       # reserved at the end for final eval + saving + sample generation

# ---- data ----
DATASET, SUBSET = "HuggingFaceFW/fineweb-edu", "sample-10BT"
VOCAB_SIZE = 32000
TOK_TRAIN_DOCS = 150_000
TARGET_TOKENS = 1_000_000_000
VAL_TOKENS = 2_000_000
EOS, PAD = "<|endoftext|>", "<|pad|>"

# ---- model (~100M params) ----
SEQ_LEN = 1024
N_LAYER, N_HEAD, N_KV_HEAD, D_MODEL, D_FF = 12, 12, 4, 768, 2048

# ---- optimisation (per-GPU micro batch; global batch = MICRO_BS * ACCUM * world * SEQ_LEN tokens) ----
MICRO_BS = int(os.environ.get("MICRO_BS", "4"))   # per GPU. 8 OOMs on a 14.5 GiB T4 (13.2 GiB at step 1, before Adam state)
ACCUM = 8                    # informational only; real accum = GLOBAL_BATCH_TOKENS / (MICRO_BS*SEQ_LEN*world)
GLOBAL_BATCH_TOKENS = 65_536 # same global batch whether 2 GPUs (accum 4) or the 1-GPU fallback (accum 8)
COMM_TIMEOUT_MIN = 15        # a hung NCCL collective aborts with an error after this, instead of hanging forever
PEAK_LR, MIN_LR_FRAC = 1e-3, 0.1
WARMUP_STEPS = 400
WEIGHT_DECAY, BETAS, CLIP = 0.1, (0.9, 0.95), 1.0
LOG_EVERY, EVAL_EVERY, EVAL_BATCHES = 50, 500, 20
CKPT_EVERY_MIN = 30
SEED = 1337
TRAIN_SECONDS_OVERRIDE = None

if SMOKE:
    VOCAB_SIZE, TOK_TRAIN_DOCS, TARGET_TOKENS, VAL_TOKENS = 1000, 2000, 400_000, 20_000
    SEQ_LEN, N_LAYER, N_HEAD, N_KV_HEAD, D_MODEL, D_FF = 64, 2, 4, 2, 64, 128
    MICRO_BS, ACCUM, WARMUP_STEPS, LOG_EVERY, EVAL_EVERY, EVAL_BATCHES = 4, 2, 5, 5, 20, 3
    TRAIN_SECONDS_OVERRIDE = 25
    GLOBAL_BATCH_TOKENS = 1024
    DATA, CKPT, FINAL = WORK / "smoke_data", WORK / "smoke_ckpt", WORK / "smoke_final"

for d in (WORK, DATA): d.mkdir(parents=True, exist_ok=True)

def wall_elapsed(): return time.time() - T0
def log(*a):
    if IS_MAIN: print(f"[{wall_elapsed()/3600:5.2f}h]", *a, flush=True)

# ------------------------------------------------------------------ data
def hf_login():
    try:
        from kaggle_secrets import UserSecretsClient
        t = UserSecretsClient().get_secret("HF_TOKEN")
        if t:
            from huggingface_hub import login
            login(token=t); log("Logged in to Hugging Face")
    except Exception:
        log("No HF_TOKEN secret - continuing anonymously (FineWeb-Edu is public). "
            "Elsewhere run `huggingface-cli login` in a terminal.")

def doc_stream():
    if SMOKE:
        import random
        rng = random.Random(0)
        words = [f"w{i}" for i in range(300)] + ["the", "a", "model", "learns", "to", "predict", "tokens"]
        while True:
            yield " ".join(rng.choice(words) for _ in range(rng.randint(40, 200)))
    from datasets import load_dataset
    seen, fails = 0, 0
    while True:
        try:
            ds = load_dataset(DATASET, name=SUBSET, split="train", streaming=True)
            if seen: ds = ds.skip(seen)              # resume after a network failure
            for ex in ds:
                seen += 1; fails = 0
                yield ex["text"]
            return
        except Exception as e:
            fails += 1
            if fails > 8: raise
            wait = min(120, 5 * 2 ** fails)
            log(f"Stream error ({type(e).__name__}: {str(e)[:120]}); retry {fails}/8 in {wait}s")
            time.sleep(wait)

def get_tokenizer():
    from transformers import PreTrainedTokenizerFast
    tdir = FINAL / "tokenizer"
    if (tdir / "tokenizer.json").exists():
        return PreTrainedTokenizerFast.from_pretrained(str(tdir))
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    log(f"Training {VOCAB_SIZE}-token byte-level BPE on {TOK_TRAIN_DOCS} docs ...")
    tk = Tokenizer(models.BPE())
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tk.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=[EOS, PAD], min_frequency=2,
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)
    tk.train_from_iterator(itertools.islice(doc_stream(), TOK_TRAIN_DOCS), trainer=trainer)
    fast = PreTrainedTokenizerFast(tokenizer_object=tk, bos_token=EOS, eos_token=EOS, pad_token=PAD,
                                   model_max_length=SEQ_LEN)
    tdir.mkdir(parents=True, exist_ok=True)
    fast.save_pretrained(str(tdir))
    log("Tokenizer saved ->", tdir, "| vocab:", len(fast))
    return fast

TRAIN_BIN, VAL_BIN, META = DATA / "train.bin", DATA / "val.bin", DATA / "meta.json"

def build_tokens(tokenizer):
    # A metadata file alone is not enough: interrupted prep runs can leave
    # metadata without usable token binaries.
    if META.exists() and TRAIN_BIN.exists() and VAL_BIN.exists():
        try:
            meta = json.loads(META.read_text())
            if TRAIN_BIN.stat().st_size >= 2 and VAL_BIN.stat().st_size >= 2:
                log("Token files already exist:", meta)
                return
        except (OSError, json.JSONDecodeError):
            pass
        log("Cached token data is incomplete or invalid; rebuilding it.")
        for path in (TRAIN_BIN, VAL_BIN, META):
            try:
                path.unlink()
            except FileNotFoundError:
                pass

    eos_id = tokenizer.convert_tokens_to_ids(EOS)
    if eos_id is None or eos_id < 0 or eos_id >= len(tokenizer):
        raise ValueError(f"Tokenizer returned an invalid EOS token ID: {eos_id}")

    log(f"Pre-tokenizing up to {TARGET_TOKENS/1e6:.0f}M tokens (streaming) ...")
    os.environ["TOKENIZERS_PARALLELISM"] = "true"
    total, train_total, val_total, buf, nflush = 0, 0, 0, [], 0
    t = time.time()

    with open(VAL_BIN, "wb") as fv, open(TRAIN_BIN, "wb") as ft:
        def write_tokens(ids):
            nonlocal total, train_total, val_total
            for token_ids in ids:
                if total >= TARGET_TOKENS:
                    break
                arr = np.asarray(token_ids + [eos_id], dtype=np.uint16)
                if arr.size == 0:
                    continue

                # Keep validation close to VAL_TOKENS instead of assigning
                # an entire large batch to validation.
                val_remaining = max(0, VAL_TOKENS - val_total)
                take_val = min(int(arr.size), val_remaining)
                if take_val:
                    fv.write(arr[:take_val].tobytes())
                    val_total += take_val
                    total += take_val
                if take_val < arr.size and total < TARGET_TOKENS:
                    take_train = min(int(arr.size - take_val), TARGET_TOKENS - total)
                    if take_train:
                        ft.write(arr[take_val:take_val + take_train].tobytes())
                        train_total += take_train
                        total += take_train

        for text in doc_stream():
            if not isinstance(text, str) or not text.strip():
                continue
            buf.append(text)
            if len(buf) >= 1000:
                ids = tokenizer(buf, add_special_tokens=False)["input_ids"]
                write_tokens(ids)
                buf = []
                nflush += 1
                if nflush % 50 == 0:
                    elapsed = max(time.time() - t, 1e-6)
                    log(f"  {total/1e6:7.1f}M tokens ({total/elapsed/1e3:.0f}k tok/s)")
                if total >= TARGET_TOKENS:
                    break

        if buf and total < TARGET_TOKENS:
            ids = tokenizer(buf, add_special_tokens=False)["input_ids"]
            write_tokens(ids)

    if train_total == 0:
        raise RuntimeError(
            "Tokenization produced no training tokens. Check dataset access and "
            "the tokenizer/data configuration."
        )
    if val_total < SEQ_LEN + 2:
        raise RuntimeError(
            f"Validation split is too small ({val_total} tokens); need at least "
            f"{SEQ_LEN + 2}. Increase the available data or validation token count."
        )

    META.write_text(json.dumps({
        "total_tokens": total,
        "train_tokens": train_total,
        "validation_tokens": val_total,
        "target_tokens": TARGET_TOKENS,
    }, indent=2))
    log(f"Done: {total/1e6:.1f}M tokens in {(time.time()-t)/60:.1f} min")

class Loader:
    """Random-window sampler over a uint16 memmap with a background prefetch thread."""
    def __init__(self, path, bs, seq, seed, prefetch=6, device_index=None):
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(
                f"Token file not found: {path}. Run `python train_pipeline.py prep` first."
            )
        if bs < 1 or seq < 1 or prefetch < 1:
            raise ValueError("bs, seq, and prefetch must all be positive integers.")

        self.data = np.memmap(path, dtype=np.uint16, mode="r")
        if len(self.data) <= seq + 1:
            self.data._mmap.close()
            raise ValueError(
                f"Token file {path} contains {path.stat().st_size // 2} tokens; "
                f"at least {seq + 2} are required for sequence length {seq}."
            )

        self.bs, self.seq = bs, seq
        self.device_index = device_index
        self.rng = np.random.default_rng(seed)
        self.q = queue.Queue(maxsize=prefetch)
        self.error = None
        self.thread = threading.Thread(target=self._work, daemon=True)
        self.thread.start()

    def _work(self):
        try:
            pin = torch.cuda.is_available()
            if pin and self.device_index is not None:
                torch.cuda.set_device(self.device_index)   # CUDA current-device is per-thread
            max_start = len(self.data) - self.seq
            while True:
                ix = self.rng.integers(0, max_start, size=self.bs)
                x = torch.from_numpy(
                    np.stack([self.data[i:i + self.seq] for i in ix]).astype(np.int64)
                )
                if pin:
                    x = x.pin_memory()
                self.q.put(x)
        except BaseException as exc:
            self.error = exc
            # Blocking put is intentional: it guarantees a waiting consumer
            # receives a sentinel even if the queue was full at the failure.
            self.q.put(None)

    def get(self):
        batch = self.q.get()
        if batch is None:
            raise RuntimeError("Background token-loader thread failed.") from self.error
        return batch

# ------------------------------------------------------------------ prep mode
def run_prep():
    hf_login()
    tok = get_tokenizer()
    assert len(tok) <= 65535
    build_tokens(tok)
    print("PREP_DONE", flush=True)

# ------------------------------------------------------------------ train mode
def run_train():
    from transformers import LlamaConfig, LlamaForCausalLM, AutoModelForCausalLM, AutoTokenizer, PreTrainedTokenizerFast
    from torch.nn.parallel import DistributedDataParallel as DDP

    world = int(os.environ.get("WORLD_SIZE", 1)); local = int(os.environ.get("LOCAL_RANK", 0))
    cuda = torch.cuda.is_available()
    print(f"[rank {RANK}] worker started | pid {os.getpid()} | world={world} | cuda={cuda} | "
          f"gpus={torch.cuda.device_count() if cuda else 0}", flush=True)
    if cuda: torch.cuda.set_device(local)                    # before any NCCL call
    device = f"cuda:{local}" if cuda else "cpu"
    if world > 1:
        t_init = time.time()
        dist.init_process_group("nccl" if cuda else "gloo", timeout=timedelta(minutes=COMM_TIMEOUT_MIN))
        probe = torch.ones(1, device=device); dist.all_reduce(probe)   # proves GPU<->GPU comms work
        assert int(probe.item()) == world, "all_reduce probe returned a wrong value"
        log(f"Process group ready, all_reduce probe OK ({time.time()-t_init:.1f}s)")
    accum = max(1, round(GLOBAL_BATCH_TOKENS / (MICRO_BS * SEQ_LEN * world)))

    tokenizer_dir = FINAL / "tokenizer"
    if not (tokenizer_dir / "tokenizer.json").exists():
        raise FileNotFoundError(
            f"Tokenizer not found at {tokenizer_dir}. Run `python train_pipeline.py prep` first."
        )
    for token_file in (TRAIN_BIN, VAL_BIN):
        if not token_file.exists() or token_file.stat().st_size < 2:
            raise FileNotFoundError(
                f"Token data is missing or empty: {token_file}. "
                "Run `python train_pipeline.py prep` first."
            )

    tokenizer = PreTrainedTokenizerFast.from_pretrained(str(tokenizer_dir))
    EOS_ID, PAD_ID = tokenizer.convert_tokens_to_ids(EOS), tokenizer.convert_tokens_to_ids(PAD)
    if EOS_ID is None or PAD_ID is None or EOS_ID < 0 or PAD_ID < 0:
        raise ValueError("The saved tokenizer is missing the configured EOS or PAD token.")

    torch.manual_seed(SEED)                                   # identical init on all ranks
    config = LlamaConfig(
        vocab_size=len(tokenizer), hidden_size=D_MODEL, intermediate_size=D_FF,
        num_hidden_layers=N_LAYER, num_attention_heads=N_HEAD, num_key_value_heads=N_KV_HEAD,
        max_position_embeddings=SEQ_LEN, rms_norm_eps=1e-5, rope_theta=10000.0, tie_word_embeddings=True,
        bos_token_id=EOS_ID, eos_token_id=EOS_ID, pad_token_id=PAD_ID, use_cache=False, initializer_range=0.02)
    config._attn_implementation = "sdpa"      # mem-efficient SDPA kernel works on T4 (FlashAttention needs Ampere+)
    raw = LlamaForCausalLM(config)
    for n, p in raw.named_parameters():       # GPT-2 style scaled init on residual output projections
        if n.endswith("o_proj.weight") or n.endswith("down_proj.weight"):
            torch.nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * N_LAYER))
    raw.to(device)
    n_params = sum(p.numel() for p in raw.parameters())
    tok_per_step = MICRO_BS * accum * SEQ_LEN * world
    log(f"Model: {n_params/1e6:.1f}M params | per-GPU batch {MICRO_BS} x accum {accum} x {world} GPU(s) | tokens/step={tok_per_step:,}")
    model = (DDP(raw, device_ids=[local], broadcast_buffers=False) if cuda else DDP(raw, broadcast_buffers=False)) if world > 1 else raw

    decay = [p for p in raw.parameters() if p.dim() >= 2]
    no_decay = [p for p in raw.parameters() if p.dim() < 2]
    kw = dict(lr=PEAK_LR, betas=BETAS, eps=1e-8)
    if cuda: kw["fused"] = True
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY},
                             {"params": no_decay, "weight_decay": 0.0}], **kw)
    try:    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    except Exception: scaler = torch.cuda.amp.GradScaler(enabled=cuda)

    # ---- resume
    step, tokens_seen, elapsed_prev = 0, 0, 0.0
    ck_path = CKPT / "state.pt"
    if ck_path.exists():
        st = torch.load(ck_path, map_location="cpu", weights_only=False)
        raw.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); scaler.load_state_dict(st["scaler"])
        step, tokens_seen, elapsed_prev = st["step"], st["tokens_seen"], st["elapsed"]
        log(f"RESUMED from step {step} ({elapsed_prev/3600:.2f}h of training already done)")
    CKPT.mkdir(parents=True, exist_ok=True)

    wall_deadline = WALL_LIMIT_H * 3600 - FINAL_RESERVE_MIN * 60
    TRAIN_SECONDS = TRAIN_SECONDS_OVERRIDE or (elapsed_prev + (wall_deadline - wall_elapsed()))
    log(f"Training budget: {TRAIN_SECONDS/3600:.2f}h total (LR schedule follows this clock)")

    def lr_at(progress, step):
        if step < WARMUP_STEPS: return PEAK_LR * (step + 1) / WARMUP_STEPS
        p = min(1.0, progress)
        return PEAK_LR * (MIN_LR_FRAC + (1 - MIN_LR_FRAC) * 0.5 * (1 + math.cos(math.pi * p)))

    train_loader = Loader(TRAIN_BIN, MICRO_BS, SEQ_LEN, seed=SEED + 1000 * RANK + step,
                          device_index=local if cuda else None)
    val_loader = Loader(VAL_BIN, MICRO_BS, SEQ_LEN, seed=999 + RANK, prefetch=2, device_index=local if cuda else None)
    amp = lambda: torch.autocast("cuda" if cuda else "cpu", dtype=torch.float16, enabled=cuda)

    @torch.no_grad()
    def evaluate():
        raw.eval(); tot = torch.zeros((), device=device)
        for _ in range(EVAL_BATCHES):
            x = val_loader.get().to(device, non_blocking=True)
            with amp(): tot += raw(input_ids=x, labels=x).loss.float()
        tot /= EVAL_BATCHES
        if world > 1: dist.all_reduce(tot); tot /= world
        raw.train(); return tot.item()

    def save_ckpt(elapsed):
        tmp = ck_path.with_suffix(".tmp")
        torch.save({"model": raw.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                    "step": step, "tokens_seen": tokens_seen, "elapsed": elapsed}, tmp)
        os.replace(tmp, ck_path)

    # ---- train loop (rank 0 owns the clock; decisions are broadcast so all ranks stay in lock-step)
    raw.train()
    sess_start, last_ckpt = time.time(), time.time()
    elapsed_now = lambda: elapsed_prev + (time.time() - sess_start)
    t_log, tok_log, hist = time.time(), 0, []
    loss_acc = torch.zeros((), device=device)
    first_step_pending = True
    log("Entering training loop ...")

    while True:
        ctl = torch.tensor([elapsed_now(), float(time.time() - last_ckpt > CKPT_EVERY_MIN * 60),
                            float(wall_elapsed() >= wall_deadline)], device=device, dtype=torch.float64)
        if world > 1: dist.broadcast(ctl, 0)
        el, do_ckpt, wall_stop = ctl.tolist()
        if el >= TRAIN_SECONDS or wall_stop: break

        lr = lr_at(el / TRAIN_SECONDS, step)
        for g in opt.param_groups: g["lr"] = lr

        for micro in range(accum):
            x = train_loader.get().to(device, non_blocking=True)
            sync = contextlib.nullcontext() if (world == 1 or micro == accum - 1) else model.no_sync()
            with sync:
                with amp(): loss = model(input_ids=x, labels=x).loss / accum
                scaler.scale(loss).backward()
            loss_acc += loss.detach().float()
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw.parameters(), CLIP)
        scaler.step(opt); scaler.update()
        opt.zero_grad(set_to_none=True)

        step += 1; tokens_seen += tok_per_step; tok_log += tok_per_step

        if first_step_pending:                      # the launcher watches for this exact marker
            first_step_pending = False
            if cuda: torch.cuda.synchronize()
            mem = f" | peak GPU mem {torch.cuda.max_memory_allocated()/2**30:.1f} GiB" if cuda else ""
            log(f"FIRST_STEP_OK | step {step} | loss {loss_acc.item():.3f}{mem}")

        if step % LOG_EVERY == 0:
            if world > 1: dist.all_reduce(loss_acc); loss_acc /= world
            l = loss_acc.item() / LOG_EVERY; loss_acc.zero_()
            dt = time.time() - t_log
            log(f"step {step:6d} | loss {l:.4f} | lr {lr:.2e} | gnorm {float(gnorm):.2f} | "
                f"{tok_log/dt/1e3:5.1f}k tok/s (all GPUs) | {tokens_seen/1e6:7.1f}M tok | "
                f"train {el/3600:.2f}h / {TRAIN_SECONDS/3600:.2f}h")
            t_log, tok_log = time.time(), 0
        if step % EVAL_EVERY == 0:
            v = evaluate()
            hist.append({"step": step, "val_loss": v, "ppl": math.exp(v), "tokens": tokens_seen})
            log(f"   >>> val loss {v:.4f} | ppl {math.exp(v):.2f}")
        if do_ckpt:
            if IS_MAIN: save_ckpt(elapsed_now()); log("   checkpoint saved")
            last_ckpt = time.time()

    log(f"Training loop finished at step {step} | {tokens_seen/1e6:.0f}M tokens")

    # ---- final eval + save (rank 0)
    final_val = evaluate()
    log(f"FINAL val loss {final_val:.4f} | ppl {math.exp(final_val):.2f}")
    if IS_MAIN:
        raw.config.use_cache = True
        FINAL.mkdir(parents=True, exist_ok=True)
        raw.save_pretrained(FINAL, safe_serialization=True)
        tokenizer.save_pretrained(FINAL)          # tokenizer sits next to the model: one folder to load from
        (FINAL / "training_summary.json").write_text(json.dumps({
            "params_M": n_params / 1e6, "steps": step, "tokens_seen": tokens_seen, "world_size": world,
            "final_val_loss": final_val, "final_ppl": math.exp(final_val), "train_hours": elapsed_now() / 3600,
            "eval_history": hist, "dataset": f"{DATASET}/{SUBSET}", "seq_len": SEQ_LEN}, indent=2))
        log("Saved ->", FINAL, sorted(p.name for p in FINAL.iterdir()))

        # standalone reload test
        del opt, model
        tk2 = AutoTokenizer.from_pretrained(FINAL)
        m2 = AutoModelForCausalLM.from_pretrained(FINAL, torch_dtype=torch.float16 if cuda else torch.float32).to(device).eval()
        enc = tk2("The water cycle is" if not SMOKE else "the model learns", return_tensors="pt").to(device)
        with torch.no_grad():
            out = m2.generate(**enc, max_new_tokens=60, do_sample=True, temperature=0.8, top_k=50,
                              repetition_penalty=1.1, pad_token_id=tk2.pad_token_id)
        print("\n--- SAMPLE ---\n" + tk2.decode(out[0], skip_special_tokens=True), flush=True)
        shutil.rmtree(CKPT, ignore_errors=True)    # resume checkpoint not needed anymore
        log("ALL DONE. Total wall time: %.2fh" % (wall_elapsed() / 3600))
    if world > 1:
        dist.barrier(); dist.destroy_process_group()

def _hard_exit(code=0):
    """Flush and leave immediately: non-daemon library threads (HF streaming, pyarrow) must never block exit."""
    if "ipykernel" in sys.modules: return
    sys.stdout.flush(); sys.stderr.flush()
    os._exit(code)

def main(mode=None):
    """CLI entry point; when pasted into a notebook cell, require an explicit mode."""
    if mode is None:
        # A notebook kernel's argv contains ipykernel_launcher and often -f <connection.json>.
        # Do not accidentally start single-process training when the whole script is run as a cell.
        if "ipykernel" in sys.modules:
            if len(sys.argv) > 1 and sys.argv[1] in ("prep", "train"):
                mode = sys.argv[1]
            else:
                print(
                    "Training script loaded in a notebook; no work was started. "
                    "Run the notebook's PREP cell first, then launch training with torchrun."
                )
                return
        else:
            mode = sys.argv[1] if len(sys.argv) > 1 else "train"

    if mode == "prep":
        run_prep(); _hard_exit(0)
    elif mode == "train":
        run_train(); _hard_exit(0)
    else:
        raise ValueError(f"Unknown mode: {mode!r}. Use 'prep' or 'train'.")


if __name__ == "__main__":
    main()

Writing /kaggle/working/train_pipeline.py


## Launcher helper (streams output, watchdog, kills hangs)

In [3]:
import os, sys, time, signal, subprocess, threading, queue
from pathlib import Path
import psutil

KERNEL_T0 = psutil.Process().create_time()          # session clock = kernel start
SCRIPT = "/kaggle/working/train_pipeline.py"
RUN_LOG = "/kaggle/working/run_log.txt"
WALL_LIMIT_H = 10.5                                  # keep identical to the script's WALL_LIMIT_H

def run_streamed(cmd, env_extra=None, ready_marker=None, startup_timeout=None,
                 stall_timeout=None, done_marker=None, done_grace=60):
    """Run cmd, stream its output into this cell, and kill it if it goes silent / hangs.
    Returns dict(rc, ready, done, reason)."""
    env = os.environ.copy()
    env.update(env_extra or {})
    env["PYTHONUNBUFFERED"] = "1"
    env["SESSION_T0"] = str(KERNEL_T0)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, env=env, start_new_session=True)
    q = queue.Queue()
    def _reader():
        for line in p.stdout: q.put(line)
        q.put(None)
    threading.Thread(target=_reader, daemon=True).start()

    start = last = time.time()
    ready, done_at, reason, oom = ready_marker is None, None, None, False
    try:
      with open(RUN_LOG, "a") as lf:
        while True:
            try: line = q.get(timeout=5)
            except queue.Empty: line = ""
            now = time.time()
            if line is None: break                                   # child closed its output -> exited
            if line:
                print(line, end="", flush=True); lf.write(line); lf.flush(); last = now
                if not ready and ready_marker in line: ready = True
                if "OutOfMemoryError" in line or "CUDA out of memory" in line: oom = True
                if done_marker and done_marker in line and done_at is None: done_at = now
            if not ready and startup_timeout and now - start > startup_timeout:
                reason = f"no '{ready_marker}' within {startup_timeout}s"; break
            if ready and stall_timeout and now - last > stall_timeout:
                reason = f"no output for {stall_timeout}s (stalled)"; break
            if done_at and now - done_at > done_grace and p.poll() is None:
                reason = "finished but process did not exit"; break
    except BaseException:
        try: os.killpg(p.pid, signal.SIGKILL)
        except ProcessLookupError: pass
        raise
    if reason:
        print(f"\n[launcher] killing process group: {reason}", flush=True)
        try: os.killpg(p.pid, signal.SIGKILL)
        except ProcessLookupError: pass
    rc = p.wait()
    subprocess.run(["pkill", "-9", "-f", "train_pipeline.py"], check=False)   # leftovers holding GPU memory
    time.sleep(3)
    return {"rc": rc, "ready": ready, "done": done_at is not None, "reason": reason, "oom": oom}

## PREP - tokenizer + data (~25-30 min on a fresh session; instant if already built)

In [4]:
# Tokenizer + 1B-token dataset (skipped quickly if already built in this session).
DATA_DIR = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/kaggle/working/data")
META = DATA_DIR / "meta.json"
for attempt in range(1, 4):
    r = run_streamed([sys.executable, "-u", SCRIPT, "prep"], done_marker="PREP_DONE", done_grace=45, stall_timeout=900)
    if (r["done"] or r["rc"] == 0) and META.exists():
        print("\nPREP OK"); break
    print(f"\nPREP attempt {attempt} failed: {r}")
else:
    raise RuntimeError("PREP failed 3 times - see log above")

[ 0.00h] Logged in to Hugging Face
[ 0.01h] Training 32000-token byte-level BPE on 150000 docs ...
[ 0.05h] Tokenizer saved -> /kaggle/working/final_model/tokenizer | vocab: 32000
[ 0.05h] Pre-tokenizing up to 1000M tokens (streaming) ...
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1078 > 1024). Running this sequence through the model will result in indexing errors
[ 0.07h]      53.0M tokens (667k tok/s)
[ 0.09h]     105.1M tokens (695k tok/s)
[ 0.11h]     156.7M tokens (703k tok/s)
[ 0.13h]     207.5M tokens (701k tok/s)
[ 0.15h]     258.8M tokens (705k tok/s)
[ 0.17h]     310.0M tokens (709k tok/s)
[ 0.19h]     362.2M tokens (708k tok/s)
[ 0.21h]     415.3M tokens (708k tok/s)
[ 0.23h]     468.3M tokens (710k tok/s)
[ 0.25h]     521.1M tokens (711k tok/s)
[ 0.27h]     573.4M tokens (710k tok/s)
[ 0.29h]     625.1M tokens (711k tok/s)
[ 0.31h]     677.2M tokens (713k tok/s)
[ 0.33h]     729.7M tokens (714k tok/s)
[ 

## TRAIN - DDP on both T4s (self-healing)

In [5]:
# Self-healing launcher: OOM -> halve per-GPU micro-batch (global batch stays 65,536 tokens);
# startup hang/crash -> next NCCL config -> 1 GPU; mid-run death -> resume from checkpoint.
subprocess.run(["pkill", "-9", "-f", "train_pipeline.py"], check=False)   # clear any orphaned workers
LADDER = [
    ("2xT4, NCCL default",           2, {}),
    ("2xT4, NCCL P2P off",           2, {"NCCL_P2P_DISABLE": "1"}),
    ("2xT4, NCCL P2P+SHM off",       2, {"NCCL_P2P_DISABLE": "1", "NCCL_SHM_DISABLE": "1", "NCCL_IB_DISABLE": "1"}),
    ("1xT4 fallback (same batch)",   1, {}),
]
idx, retries, micro_bs, ok = 0, 0, 4, False
while idx < len(LADDER) and not ok:
    if (time.time() - KERNEL_T0) / 3600 > WALL_LIMIT_H - 0.5:
        print("Not enough wall-clock budget left to start another attempt."); break
    name, nproc, extra = LADDER[idx]
    print(f"\n===== TRAIN attempt: {name} | per-GPU micro-batch {micro_bs} =====", flush=True)
    cmd = [sys.executable, "-u", "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={nproc}", SCRIPT, "train"]
    env = {"NCCL_DEBUG": "WARN", "MICRO_BS": str(micro_bs), "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", **extra}
    r = run_streamed(cmd, env, ready_marker="FIRST_STEP_OK", startup_timeout=420, stall_timeout=1500,
                     done_marker="ALL DONE", done_grace=180)
    if r["done"] or (r["rc"] == 0 and r["ready"]):
        ok = True
    elif r["oom"] and micro_bs > 1:
        micro_bs //= 2; print(f"CUDA OOM -> retrying with per-GPU micro-batch {micro_bs}")
    elif r["ready"] and retries < 2:
        retries += 1; print(f"Training died mid-run ({r}); resuming from checkpoint (retry {retries}/2)")
    else:
        retries = 0; idx += 1; print(f"Attempt failed to start ({r}); trying next config")
print("\nTRAINING FINISHED OK" if ok else "\nTRAINING DID NOT COMPLETE - see log above")


===== TRAIN attempt: 2xT4, NCCL default | per-GPU micro-batch 4 =====

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[rank 0] worker started | pid 78 | world=2 | cuda=True | gpus=2
[rank 1] worker started | pid 79 | world=2 | cuda=True | gpus=2
NCCL version 2.31.2+cuda13.3
[ 0.44h] Process group ready, all_reduce probe OK (0.8s)
[ 0.44h] Model: 100.1M params | per-GPU batch 4 x accum 8 x 2 GPU(s) | tokens/step=65,536
[ 0.44h] Training budget: 9.86h total (LR schedule follows this clock)
[ 0.44h] Entering training loop ...
[ 0.45h] FIRST_STEP_OK | step 1 | loss 10.528 | peak GPU mem 7.2 GiB
[ 0.51h] step     50 | loss 8.6962 | lr 1.25e-04 | gnorm 0.88 |  14.9k tok/s (all GPUs) |     3.3M tok | train 0.06h / 9.86h
[ 0.58h] step    100 | loss 6

In [6]:
!ls -lh /kaggle/working/final_model /kaggle/working/final_model/tokenizer 2>/dev/null; du -sh /kaggle/working

/kaggle/working/final_model:
total 385M
-rw-r--r-- 1 root root  720 Oct 19 19:12 config.json
-rw-r--r-- 1 root root  217 Oct 19 19:12 generation_config.json
-rw------- 1 root root 382M Oct 19 19:12 model.safetensors
drwxr-xr-x 2 root root 4.0K Oct 19 08:56 tokenizer
-rw-r--r-- 1 root root  239 Oct 19 19:12 tokenizer_config.json
-rw-r--r-- 1 root root 2.2M Oct 19 19:12 tokenizer.json
-rw-r--r-- 1 root root 2.1K Oct 19 19:12 training_summary.json

/kaggle/working/final_model/tokenizer:
total 2.2M
-rw-r--r-- 1 root root  190 Oct 19 08:56 tokenizer_config.json
-rw-r--r-- 1 root root 2.2M Oct 19 08:56 tokenizer.json
2.3G	/kaggle/working
